Phase 1: NBA Spatial Data Extraction. This notebook connects to the live NBA REST API to extract raw shot coordinates for the Golden State Warriors. It performs initial ETL (Extract, Transform, Load) to clean the JSON response into a structured DataFrame.

In [8]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster, shotchartdetail
import pandas as pd
import time
from google.colab import files

print("Tools loaded successfully.")

Tools loaded successfully.


In [9]:
# Find GSW ID
nba_teams = teams.get_teams()
gsw_id = [team for team in nba_teams if team['abbreviation'] == 'GSW'][0]['id']

# Get the Roster
print("Fetching the 2021-22 Championship Roster...")
roster = commonteamroster.CommonTeamRoster(team_id=gsw_id, season='2021-22')
df_roster = roster.get_data_frames()[0]

# Create the checklist
players_to_fetch = df_roster[['PLAYER_ID', 'PLAYER']]
print(f"Target Acquired: {len(players_to_fetch)} players found on the roster.")

Fetching the 2021-22 Championship Roster...
Target Acquired: 17 players found on the roster.


API calls to the NBA server frequently time out. This loop incorporates a try/except block with a 3-attempt retry limit and rate-limiting (time.sleep) to ensure the pipeline doesn't crash during server lag.

In [10]:
all_shots_data = []

for index, row in players_to_fetch.iterrows():
    player_id = row['PLAYER_ID']
    player_name = row['PLAYER']

    print(f"Extracting spatial data for: {player_name}...")

    success = False
    attempts = 0

    while not success and attempts < 3:
        try:
            shot_data = shotchartdetail.ShotChartDetail(
                team_id=gsw_id,
                player_id=player_id,
                context_measure_simple='FGA',
                season_nullable='2021-22',
                season_type_all_star='Regular Season',
                timeout=60
            )

            df_player_shots = shot_data.get_data_frames()[0]
            all_shots_data.append(df_player_shots)
            success = True

        except Exception as e:
            attempts += 1
            print(f"  -> Server timed out (Attempt {attempts}/3). Pausing for 5 seconds...")
            time.sleep(5)

    time.sleep(1)

print("\nAll player data successfully extracted and stored in memory!")

Extracting spatial data for: Gary Payton II...
Extracting spatial data for: Jonathan Kuminga...
Extracting spatial data for: Damion Lee...
Extracting spatial data for: Chris Chiozza...
Extracting spatial data for: Jordan Poole...
Extracting spatial data for: Moses Moody...
Extracting spatial data for: Kevon Looney...
Extracting spatial data for: Nemanja Bjelica...
Extracting spatial data for: Andre Iguodala...
Extracting spatial data for: Klay Thompson...
Extracting spatial data for: Quinndary Weatherspoon...
Extracting spatial data for: Andrew Wiggins...
Extracting spatial data for: Draymond Green...
Extracting spatial data for: Stephen Curry...
Extracting spatial data for: Otto Porter Jr....
Extracting spatial data for: James Wiseman...
Extracting spatial data for: Juan Toscano-Anderson...

All player data successfully extracted and stored in memory!


The raw API response contains redundant metadata (game IDs, arena names). This step filters the dataset down strictly to the analytical features needed for spatial mapping and Expected Value calculations.

In [11]:
# Glue it together
df_master = pd.concat(all_shots_data, ignore_index=True)

# Clean it
columns_to_keep = [
    'PLAYER_NAME', 'GAME_DATE', 'ACTION_TYPE', 'SHOT_TYPE',
    'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_DISTANCE',
    'LOC_X', 'LOC_Y', 'SHOT_MADE_FLAG'
]
df_clean = df_master[columns_to_keep]

# Export it
filename = "GSW_2022_Roster_Shots.csv"
df_clean.to_csv(filename, index=False)
print(f"Master dataset compiled: {len(df_clean)} total shots.")

files.download(filename)

Master dataset compiled: 7081 total shots.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
df_clean.head()


,PLAYER_NAME,GAME_DATE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_MADE_FLAG
0,Gary Payton II,20211019,Floating Jump shot,2PT Field Goal,In The Paint (Non-RA),Center(C),6,-29,55,0
1,Gary Payton II,20211024,Alley Oop Dunk Shot,2PT Field Goal,Restricted Area,Center(C),1,-10,8,1
2,Gary Payton II,20211024,Cutting Layup Shot,2PT Field Goal,Restricted Area,Center(C),2,-28,10,0
3,Gary Payton II,20211024,Jump Shot,2PT Field Goal,Restricted Area,Center(C),2,-28,6,0
4,Gary Payton II,20211024,Running Layup Shot,2PT Field Goal,Restricted Area,Center(C),1,-18,8,1
